# Experimento 2 - Variacao de carga

Objetivo: analisar como cada politica (`threshold`, `on_demand`, `hybrid`) se comporta quando a demanda cresce.

Este experimento varia intensidade de requisicoes por tres perfis:
- carga baixa
- carga media
- carga alta

E observa:
- taxa de atendimento = `served_requests / (served_requests + denied_requests)`
- eficiencia de uso = `total_consumed_bits / total_generated_bits`
- pressao sobre o sistema = `replenishment_events`
- escassez residual = `bits_available` final

In [1]:
import sys
import random
import pandas as pd

sys.path.insert(0, '/home/esdras/Wqnets/QuantumNet')

from quantumnet.topology import Network

print('Imports carregados com sucesso')

Imports carregados com sucesso


## Configuracao do experimento

Topologia fixa: `Linha` com 4 nos e 3 enlaces diretos (`(0,1)`, `(1,2)`, `(2,3)`).

Para cada combinacao de politica e nivel de carga, executamos varias repeticoes para suavizar variacao estocastica do BB84.

In [2]:
policies = ['threshold', 'on_demand', 'hybrid']
link_pairs = [(0, 1), (1, 2), (2, 3)]
minimum_stock_bits = 96
trials_per_setting = 8

# Cada perfil combina volume de requisicoes e tamanho de pedido (em bits).
load_profiles = {
    'baixa': {
        'requests_per_link': 3,
        'bit_options': [16, 24, 32],
    },
    'media': {
        'requests_per_link': 6,
        'bit_options': [24, 32, 48, 64],
    },
    'alta': {
        'requests_per_link': 10,
        'bit_options': [48, 64, 96, 128],
    },
}

print('Politicas:', policies)
print('Perfis de carga:', list(load_profiles.keys()))
print('Repeticoes por combinacao:', trials_per_setting)

Politicas: ['threshold', 'on_demand', 'hybrid']
Perfis de carga: ['baixa', 'media', 'alta']
Repeticoes por combinacao: 8


## Execucao

Cada repeticao executa o fluxo de consumo via `controller.handle_key_request(...)`, que internamente usa `request_key_from_buffer()` e atualiza as metricas de pedido/consumo/falha do enlace.

In [3]:
def run_single_trial(policy: str, load_name: str, profile: dict, trial_id: int, base_seed: int = 2026) -> dict:
    rng = random.Random(base_seed + trial_id * 100 + hash((policy, load_name)) % 10000)

    net = Network()
    net.set_ready_topology('Linha', 4)
    net.controller.set_policy(policy)

    for alice_id, bob_id in link_pairs:
        net.controller.set_minimum_stock(alice_id, bob_id, minimum_stock_bits)

    # Gera carga de requisicoes para cada enlace
    for alice_id, bob_id in link_pairs:
        for _ in range(profile['requests_per_link']):
            requested_bits = rng.choice(profile['bit_options'])
            net.controller.handle_key_request(alice_id, bob_id, requested_bits)

    # Agrega metricas de todos os enlaces
    totals = {
        'total_generated_bits': 0,
        'total_consumed_bits': 0,
        'served_requests': 0,
        'denied_requests': 0,
        'replenishment_events': 0,
        'bits_available': 0,
    }

    for alice_id, bob_id in link_pairs:
        state = net.get_qkd_link_state(alice_id, bob_id)
        totals['total_generated_bits'] += int(state['total_generated_bits'])
        totals['total_consumed_bits'] += int(state['total_consumed_bits'])
        totals['served_requests'] += int(state['served_requests'])
        totals['denied_requests'] += int(state['denied_requests'])
        totals['replenishment_events'] += int(state['replenishment_events'])
        totals['bits_available'] += int(state['bits_available'])

    served = totals['served_requests']
    denied = totals['denied_requests']
    generated = totals['total_generated_bits']
    consumed = totals['total_consumed_bits']

    service_rate = served / (served + denied) if (served + denied) > 0 else 0.0
    efficiency = consumed / generated if generated > 0 else 0.0

    return {
        'policy': policy,
        'load': load_name,
        'trial': trial_id,
        **totals,
        'service_rate': service_rate,
        'efficiency': efficiency,
    }

rows = []
for policy in policies:
    for load_name, profile in load_profiles.items():
        for trial in range(1, trials_per_setting + 1):
            rows.append(run_single_trial(policy, load_name, profile, trial))

results_df = pd.DataFrame(rows)
results_df.head()

,policy,load,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency
0,threshold,baixa,1,192,80,4,5,2,112,0.444444,0.416667
1,threshold,baixa,2,96,64,2,7,1,32,0.222222,0.666667
2,threshold,baixa,3,192,40,2,7,2,152,0.222222,0.208333
3,threshold,baixa,4,96,56,3,6,1,40,0.333333,0.583333
4,threshold,baixa,5,192,80,4,5,2,112,0.444444,0.416667


## Resultado detalhado por repeticao

In [4]:
display(results_df.sort_values(['policy', 'load', 'trial']).reset_index(drop=True))

,policy,load,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency
0,hybrid,alta,1,656,656,8,22,10,0,0.266667,1.000000
1,hybrid,alta,2,608,608,10,20,10,0,0.333333,1.000000
2,hybrid,alta,3,640,640,9,21,10,0,0.300000,1.000000
3,hybrid,alta,4,672,624,12,18,12,48,0.400000,0.928571
4,hybrid,alta,5,608,608,9,21,8,0,0.300000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
67,threshold,media,4,192,192,4,14,2,0,0.222222,1.000000
68,threshold,media,5,288,256,6,12,3,32,0.333333,0.888889
69,threshold,media,6,288,208,5,13,3,80,0.277778,0.722222
70,threshold,media,7,192,160,4,14,2,32,0.222222,0.833333


## Resumo agregado por politica e carga

Media e desvio padrao das metricas principais para enxergar tendencia de saturacao.

In [5]:
summary_df = (
    results_df
    .groupby(['policy', 'load'], as_index=False)
    .agg(
        service_rate_mean=('service_rate', 'mean'),
        service_rate_std=('service_rate', 'std'),
        efficiency_mean=('efficiency', 'mean'),
        efficiency_std=('efficiency', 'std'),
        replenishment_events_mean=('replenishment_events', 'mean'),
        bits_available_mean=('bits_available', 'mean'),
        denied_requests_mean=('denied_requests', 'mean'),
        served_requests_mean=('served_requests', 'mean'),
    )
    .sort_values(['policy', 'load'])
    .reset_index(drop=True)
)

display(summary_df)

,policy,load,service_rate_mean,service_rate_std,efficiency_mean,efficiency_std,replenishment_events_mean,bits_available_mean,denied_requests_mean,served_requests_mean
0,hybrid,alta,0.325000,0.049602,0.985119,0.028279,10.125,12.0,20.250,9.750
1,hybrid,baixa,0.861111,0.115011,0.830640,0.152901,6.375,46.0,1.250,7.750
2,hybrid,media,0.638889,0.122438,0.929433,0.084766,10.250,34.0,6.500,11.500
3,on_demand,alta,0.283333,0.102353,1.000000,0.000000,8.500,0.0,21.500,8.500
4,on_demand,baixa,0.847222,0.144719,1.000000,0.000000,7.625,0.0,1.375,7.625
5,on_demand,media,0.631944,0.078216,1.000000,0.000000,11.375,0.0,6.625,11.375
6,threshold,alta,0.058333,0.061075,0.645833,0.421990,1.500,16.0,28.250,1.750
7,threshold,baixa,0.361111,0.129441,0.442708,0.140945,1.875,107.0,5.750,3.250
8,threshold,media,0.187500,0.106812,0.706597,0.344988,1.875,36.0,14.625,3.375


## Leitura rapida: ponto de saturacao

Critrio simples usado aqui: considerar saturacao quando `service_rate_mean < 0.90`.

Se nenhuma carga cair abaixo disso, a politica sustentou o intervalo testado.

In [6]:
load_order = {'baixa': 1, 'media': 2, 'alta': 3}
summary_ordered = summary_df.assign(load_rank=summary_df['load'].map(load_order)).sort_values(['policy', 'load_rank'])

print('Resumo por politica:')
for policy in policies:
    subset = summary_ordered[summary_ordered['policy'] == policy]

    saturation_rows = subset[subset['service_rate_mean'] < 0.90]
    if len(saturation_rows) > 0:
        sat_load = saturation_rows.iloc[0]['load']
    else:
        sat_load = 'nao saturou no intervalo testado'

    print(f'\n- Politica: {policy}')
    print(f'  Primeiro sinal de saturacao: {sat_load}')

    for _, row in subset.iterrows():
        print(
            f"  Carga {row['load']}: "
            f"atendimento={row['service_rate_mean']:.3f}, "
            f"eficiencia={row['efficiency_mean']:.3f}, "
            f"reposicoes={row['replenishment_events_mean']:.1f}, "
            f"buffer_final={row['bits_available_mean']:.1f}"
        )

Resumo por politica:

- Politica: threshold
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.361, eficiencia=0.443, reposicoes=1.9, buffer_final=107.0
  Carga media: atendimento=0.188, eficiencia=0.707, reposicoes=1.9, buffer_final=36.0
  Carga alta: atendimento=0.058, eficiencia=0.646, reposicoes=1.5, buffer_final=16.0

- Politica: on_demand
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.847, eficiencia=1.000, reposicoes=7.6, buffer_final=0.0
  Carga media: atendimento=0.632, eficiencia=1.000, reposicoes=11.4, buffer_final=0.0
  Carga alta: atendimento=0.283, eficiencia=1.000, reposicoes=8.5, buffer_final=0.0

- Politica: hybrid
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.861, eficiencia=0.831, reposicoes=6.4, buffer_final=46.0
  Carga media: atendimento=0.639, eficiencia=0.929, reposicoes=10.2, buffer_final=34.0
  Carga alta: atendimento=0.325, eficiencia=0.985, reposicoes=10.1, buffer_final=12.0


## O que este experimento responde

Este cenario mostra em que ponto cada politica deixa de sustentar a rede QKD sob aumento de demanda, observando simultaneamente:
- queda de taxa de atendimento,
- mudanca na eficiencia de uso dos bits gerados,
- crescimento de eventos de reposicao,
- e nivel final de escassez residual no buffer.